In [ ]:
import os
import chromadb
from chromadb.config import Settings

client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# ลบ collection
client.delete_collection("member_resumes_simple")
print("✅ ลบ collection สำเร็จ")

# ตรวจสอบ
collections = client.list_collections()
print(f"Collections ที่เหลือ: {[c.name for c in collections]}")

In [ ]:
pip install chromadb

In [ ]:
import os
import chromadb
from chromadb.config import Settings

# เชื่อมต่อ ChromaDB
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# ดู collections
collections = client.list_collections()
print(f"Collections: {[c.name for c in collections]}")

# เข้าถึง collection
collection = client.get_collection("job_postings_simple")

# นับจำนวนข้อมูล
count = collection.count()
print(f"Total documents: {count:,}")


def print_simple_job(meta, doc, index, distance=None):
    """แสดงรายละเอียดงานแบบง่าย - รวม Position"""
    
    print(f"\n{'─'*80}")
    print(f"ลำดับที่ {index}")
    print(f"{'─'*80}")
    
    # แสดง Position
    position = meta.get('position', 'N/A')
    print(f"\n📌 ตำแหน่ง: {position}")
    
    # แสดง IDs
    print(f"\n🆔 Job Post ID: {meta.get('jobpost_id', 'N/A')}")
    print(f"   Company ID:  {meta.get('company_id', 'N/A')}")
    print(f"   Job ID:      {meta.get('job_id', 'N/A')}")
    
    # แสดง distance (ถ้ามี)
    if distance is not None:
        similarity = 1 - distance
        print(f"\n🎯 Similarity:  {similarity:.4f} ({similarity*100:.2f}%)")
    
    # แสดง detail (ตัดส่วน "ตำแหน่ง: ... |" ออก)
    print(f"\n📄 รายละเอียดงาน:")
    if doc:
        # ตัดส่วน "ตำแหน่ง: ... |" ออก ถ้ามี
        cleaned_doc = doc
        if '|' in doc and doc.strip().startswith('ตำแหน่ง:'):
            # แยกเอาเฉพาะส่วนหลัง " | "
            parts = doc.split('|', 1)
            if len(parts) > 1:
                cleaned_doc = parts[1].strip()
        
        # แสดง 500 ตัวอักษรแรก
        detail_preview = cleaned_doc[:500] + '...' if len(cleaned_doc) > 500 else cleaned_doc
        print(f"{detail_preview}")
    else:
        print("(ไม่มีรายละเอียด)")


# รายการ queries - เน้นค้นหาจากเนื้อหาใน detail
queries = [
    # ค้นหาจากหน้าที่และความรับผิดชอบ
    "ออกแบบ database เขียน API backend",
    "ดูแลระบบ server network infrastructure",
    "ประสานงานลูกค้า นำเสนอขาย",
    "วิเคราะห์ข้อมูล สร้าง report dashboard",
    "จัดการ social media content marketing",
    
    # ค้นหาจากทักษะ
    "Python Django PostgreSQL REST API",
    "React TypeScript Next.js frontend",
    "digital marketing SEO Google Ads",
    "Excel PowerBI data analysis",
    "Photoshop Illustrator graphic design",
    
    # ค้นหาจากลักษณะงาน
    "work from home remote",
    "รับเด็กจบใหม่ ไม่ต้องมีประสบการณ์",
    "งานด่วน เริ่มงานทันที",
    "เงินเดือน 30000-50000",
    "ทำงาน 5 วัน ประกันสังคม",
]

# ตั้งค่าการ query
n_results = 3  # จำนวนผลลัพธ์ที่ต้องการ

print(f"\n\n{'='*80}")
print(f"เริ่มค้นหา - จำนวน {len(queries)} queries")
print(f"{'='*80}")

# ทำการ query
for idx, query in enumerate(queries, 1):
    print(f"\n\n{'='*80}")
    print(f"Query #{idx}: {query}")
    print(f"{'='*80}")
    
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
        
        # แสดงผลลัพธ์
        if results['documents'] and results['documents'][0]:
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results.get('distances', [[None] * len(documents)])[0]
            
            for i, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), 1):
                print_simple_job(meta, doc, i, dist)
        else:
            print("\nไม่พบผลลัพธ์")
            
    except Exception as e:
        print(f"\nError: {e}")

print(f"\n\n{'='*80}")
print("เสร็จสิ้นการค้นหา")
print(f"{'='*80}")


# ฟังก์ชันเสริม: ค้นหาด้วย keyword แบบ flexible
def flexible_search(keyword, top_k=5):
    """ค้นหาแบบยืดหยุ่น แสดงผลแบบกระชับ - รวม Position"""
    print(f"\n{'='*80}")
    print(f"Flexible Search: '{keyword}'")
    print(f"{'='*80}")
    
    results = collection.query(
        query_texts=[keyword],
        n_results=top_k
    )
    
    if results['documents'] and results['documents'][0]:
        for i, (doc, meta, dist) in enumerate(zip(
            results['documents'][0], 
            results['metadatas'][0],
            results.get('distances', [[None] * top_k])[0]
        ), 1):
            similarity = (1 - dist) * 100 if dist is not None else 0
            position = meta.get('position', 'N/A')
            
            # ตัดส่วน "ตำแหน่ง: ... |" ออก
            cleaned_doc = doc
            if '|' in doc and doc.strip().startswith('ตำแหน่ง:'):
                parts = doc.split('|', 1)
                if len(parts) > 1:
                    cleaned_doc = parts[1].strip()
            
            print(f"\n{i}. 📌 {position}")
            print(f"   🆔 JobPost: {meta['jobpost_id']} | Company: {meta['company_id']}")
            print(f"   🎯 Match: {similarity:.1f}%")
            print(f"   📄 {cleaned_doc[:150]}...")
    else:
        print("ไม่พบผลลัพธ์")


# ตัวอย่างการใช้งาน flexible_search
print("\n\n" + "="*80)
print("ตัวอย่างการค้นหาแบบ Flexible")
print("="*80)

flexible_search("โปรแกรมเมอร์ Python", top_k=3)
flexible_search("marketing digital", top_k=3)
flexible_search("ไม่ต้องมีประสบการณ์", top_k=3)

In [ ]:
import os
# import chromadb
# import pandas as pd

# client = chromadb.HttpClient(
#     host=os.getenv("CHROMA_HOST", "localhost"),
#     port=int(os.getenv("CHROMA_PORT", "8000")),
#     settings=Settings(anonymized_telemetry=False)
# )

# collection = client.get_collection("job_postings_simple")

# # เนื่องจาก limit สูงสุดคือ ~10,000 ต้องดึงแบบ paginate
# all_data = []
# batch_size = 1000
# total = collection.count()

# for offset in range(0, total, batch_size):
#     print(f"Fetching {offset}-{offset+batch_size}...")
#     # ChromaDB ไม่รองรับ offset โดยตรง ต้องใช้วิธีอื่น
#     # แต่สามารถดึงทั้งหมดแล้ว slice
#     pass

# # วิธีที่ง่ายกว่า: ดึงทีละ 10,000
# data1 = collection.get(limit=10000, include=['metadatas'])
# data2 = collection.get(limit=10000, offset=10000, include=['metadatas'])  # อาจไม่รองรับ
# # หรือดึงทั้งหมดเลย (อาจใช้เวลานาน)
# all_data = collection.get(include=['metadatas', 'documents'])

# df = pd.DataFrame(all_data['metadatas'])
# df.to_csv('all_jobs.csv', index=False, encoding='utf-8-sig')
# print(f"Exported {len(df)} records")